In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
selected_features = [
    "Brand",
    "Series",
    "Thickness",
    "Weight",
    "Operating System",
    "Display Size",
    "Display Touchscreen",
    "Processor",
    "Graphic Processor",
    "RAM_Capacity_GB",
    "RAM Type",
    "SSD Capacity",
    "HDD Capacity",
    "Battery Capacity",
    "Fingerprint scanner"
]

target = "Price (Rs)"

In [3]:
numerical_features = [
    "Thickness",
    "Weight",
    "Display Size",
    "RAM_Capacity_GB",
    "SSD Capacity",
    "HDD Capacity",
    "Battery Capacity"
]

categorical_features = [
    "Brand",
    "Series",
    "Operating System",
    "Display Touchscreen",
    "Processor",
    "Graphic Processor",
    "RAM Type",
    "Fingerprint scanner"
]

In [4]:
df = pd.read_csv("../data/processed/laptop_selected.csv")
X = df.drop(columns=["Price (Rs)"])
y = df["Price (Rs)"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [6]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [7]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [8]:
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [9]:
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1))
])

In [ ]:
final_pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['Brand','Series','Thickness',...,'HDD Capacity','Battery Capacity', 'Fingerprint scanner']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. Th

In [11]:
pipeline_predictions = final_pipeline.predict(X_test)

In [13]:
final_mae = mean_absolute_error(y_test,pipeline_predictions)
final_rmse = np.sqrt(mean_squared_error(y_test,pipeline_predictions))
final_r2 = r2_score(y_test,pipeline_predictions)

print("Final Pipeline Test MAE :", final_mae)
print("Final Pipeline Test RMSE:", final_rmse)
print("Final Pipeline Test R²  :", final_r2)

Final Pipeline Test MAE : 12720.36130129627
Final Pipeline Test RMSE: 20290.92939416452
Final Pipeline Test R²  : 0.8900857912500585


In [15]:
os.makedirs("../models", exist_ok=True)

In [16]:
model_path = "../models/final_laptopwise_pipeline.joblib"
joblib.dump(final_pipeline,model_path)
print(f"Pipeline saved to: {model_path}")

Pipeline saved to: ../models/final_laptopwise_pipeline.joblib


In [17]:
print("File exists:", os.path.exists(model_path))

File exists: True


In [18]:
file_size = os.path.getsize(model_path)

print(f"File size: {file_size / 1024:.2f} KB")

File size: 53.86 KB


In [19]:
loaded_pipeline = joblib.load(model_path)

In [20]:
reloaded_predictions = loaded_pipeline.predict(X_test)

In [23]:
new_laptop = pd.DataFrame({
    "Brand": ["Lenovo"],
    "Series": ["IdeaPad"],
    "Thickness": [19.9],
    "Weight": [1.65],
    "Operating System": ["Windows 11"],
    "Display Size": [15.6],
    "Display Touchscreen": ["No"],
    "Processor": ["Intel Core i5"],
    "Graphic Processor": ["Intel Integrated Graphics"],
    "RAM_Capacity_GB": [16],
    "RAM Type": ["DDR4"],
    "SSD Capacity": [512],
    "HDD Capacity": [0],
    "Battery Capacity": [57],
    "Fingerprint scanner": ["Yes"]
})

In [24]:
print("Number of features:", new_laptop.shape[1])

Number of features: 15


In [27]:
print("Target supplied:", target in new_laptop.columns)

Target supplied: False


In [28]:
prediction = loaded_pipeline.predict(new_laptop)

In [29]:
predicted_price = prediction[0]
print(f"Predicted Laptop Price: ₹{predicted_price:,.2f}")

Predicted Laptop Price: ₹93,432.83
